# LOB Anomaly Detection: v2 Dense Autoencoder
**Kaggle Competition, Machine Learning Final Project, NYU**

## What are we actually trying to do?

We're working with **limit order book (LOB) data**, a real-time log of every buy and sell order placed on a stock exchange. The competition asks us to flag rows that represent manipulative trading: **pump-and-dump**, **layering**, and **quote stuffing**.

The hard part: we never see a labeled fraud example during training. Both the training and validation sets are 100% normal market activity. Fraud only appears, hidden, in the test set.

This is called **semi-supervised anomaly detection**. We can't build a regular supervised classifier because we have no fraud labels. Instead, we learn what *normal* market behavior looks like, then flag anything that significantly deviates from it.

**Evaluation metric**: AUROC (Area Under the ROC Curve). Explained in full in Section 8. Short version: 1.0 = perfect, 0.5 = random coin flip. The class baseline is **0.88**.

## What's in this notebook (v2)

**Carried over from v1:**
- A **z-score heuristic** that flags rows with extreme individual feature values relative to normal
- An **Isolation Forest**, an ensemble anomaly detector (**not covered in class**, so we explain the math in Section 4)

**New in v2:**
- A **Dense Autoencoder** built from an MLP backbone (**was covered in class**). Trained only on normal data; fraud rows can't be reconstructed well, so high reconstruction error means a high anomaly score.
- A **weighted ensemble** that merges all three detector scores into one final anomaly score.

*Important rule throughout: no model, scaler, or statistic is ever fit on the test set. Everything learns from normal data only.*

## Section 1: Setup

Nothing unusual here, just imports and constants. The key import is `MLPRegressor` from sklearn, which we repurpose as our autoencoder (explained in Section 5).

In [ ]:
# MLPRegressor is sklearn's multilayer perceptron — we use it as our autoencoder
from sklearn.neural_network import MLPRegressor
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)  # fixed seed so results are reproducible every run

# update these paths if your data CSVs are stored somewhere else
TRAIN_PATH = 'train_data.csv'
VAL_PATH   = 'val_data.csv'
TEST_PATH  = 'test_data.csv'
OUTPUT_DIR = './'

# the 14 raw numeric features from the LOB dataset
# we engineer ~23 more on top of these in Section 3, bringing the total to 37
NUMERIC_FEATURES = [
    'ReturnBid1', 'ReturnAsk1',
    'DerivativeReturnBid1', 'DerivativeReturnAsk1',
    'BidSize1', 'AskSize1',
    'TradeBidSize', 'TradeAskSize',
    'CancelledBidSize', 'CancelledAskSize',
    'TradeBidIndicator', 'TradeAskIndicator',
    'CancelledBidIndicator', 'CancelledAskIndicator',
]
print('Setup complete')

## Section 2: Loading the Data

We have three CSV files:

- `train_data.csv` has about 1,070,000 rows and is all normal data (FraudType = 0)
- `val_data.csv` has about 468,000 rows and is also all normal (FraudType = 0)
- `test_data.csv` has about 289,000 rows and is a mix of normal and hidden fraud

Since both train and val are entirely clean, we merge them into one big **normal pool** of about 1.5 million rows. All model fitting, scalers, and statistics are computed on this pool. The more normal data our models see, the better they learn what normal actually looks like.

The test set never touches any fitting step, it only gets scored at the end.

In [ ]:
print('Loading data...')
train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)

# merge train + val — both are all-normal, so combining them is safe
# more normal data = better coverage of what normal looks like
normal = pd.concat([train, val], ignore_index=True)

print(f'  Train    : {len(train):>10,} rows  (all normal)')
print(f'  Val      : {len(val):>10,} rows  (all normal)')
print(f'  Normal   : {len(normal):>10,} rows  combined  used for ALL fitting')
print(f'  Test     : {len(test):>10,} rows  (hidden fraud inside  never used for fitting)')

## Section 3: Feature Engineering

The dataset gives us 14 numeric features. We build a richer 37-feature matrix using three types of engineered signals:

### 1. Burst features, the strongest fraud signal we found

The `OriginalSequenceNumber` column contains values like `456789-1`, `456789-2`, `456789-3`. The number before the dash is a "base" sequence ID, and all sub-orders sharing that base were placed as a single burst. From exploratory analysis, we found that **bursts of 10 to 49 sub-orders had 175x higher mean |ReturnBid1|** than single orders. That's the clearest fraud fingerprint in the whole dataset.

We create:
- `burst_size`: how many sub-orders share the same base sequence ID
- `burst_log`: log(1 + burst_size) to compress the scale for the models
- `burst_10_49`: binary flag for whether this order is part of a 10-to-49 burst
- `burst_50plus`: binary flag for bursts of 50 or more

> **No leakage**: burst sizes are computed within each dataset independently, so test-set burst counts never influence the normal-pool statistics.

### 2. Per-symbol z-scores

Different stocks trade at different scales. A return of 0.01 might be extreme for one stock and perfectly routine for another. We compute z-scores per stock symbol using only normal-pool statistics:

$$z_j(x) = \frac{|x_j - \mu_j^{\text{train}}|}{\sigma_j^{\text{train}}}$$

where $\mu_j$ and $\sigma_j$ come from the normal pool only. We freeze these stats and apply them to test without re-fitting. We get 14 per-feature z-scores plus `max_zscore` and `mean_zscore`.

### 3. Simple magnitude features

Absolute values of bid/ask returns, the spread between them, and the maximum absolute feature value. Simple but useful for models that look at raw scale rather than relative deviation.

In [ ]:
def add_burst_features(df):
    '''
    OriginalSequenceNumber looks like "456789-2".
    We split on "-" to get the base ID and count how many sub-orders share it.
    Burst size is the strongest fraud signal we found in EDA:
    bursts of 10-49 had 175x higher |ReturnBid1| than single orders.

    Computed independently within each dataset, so there is no leakage.
    '''
    df = df.copy()
    df['base_seq']     = df['OriginalSequenceNumber'].astype(str).str.split('-').str[0]
    burst              = df.groupby('base_seq').size().rename('burst_size')
    df                 = df.join(burst, on='base_seq')
    df['burst_log']    = np.log1p(df['burst_size'])           # log scale to compress big values
    df['burst_10_49']  = ((df['burst_size'] >= 10) & (df['burst_size'] < 50)).astype(float)
    df['burst_50plus'] = (df['burst_size'] >= 50).astype(float)
    return df


def compute_training_stats(normal_df):
    '''
    Compute per-symbol mean and std from NORMAL DATA ONLY.
    We freeze these stats and apply them to the test set later.
    Re-computing stats from test data would be leakage.
    '''
    filled = normal_df[NUMERIC_FEATURES].fillna(0)
    stats  = {}
    for sym in normal_df['ExternalSymbol'].unique():
        mask = (normal_df['ExternalSymbol'] == sym).values
        sub  = filled.values[mask]
        stats[sym] = {
            col: (sub[:, j].mean(), max(sub[:, j].std(), 1e-9))
            for j, col in enumerate(NUMERIC_FEATURES)
        }
    # fallback for any stock symbol in test that never appeared in training
    stats['__global__'] = {
        col: (filled[col].mean(), max(filled[col].std(), 1e-9))
        for col in NUMERIC_FEATURES
    }
    return stats


def compute_zscores(df, stats):
    '''
    z = |value - training_mean| / training_std  (per stock symbol).
    Uses frozen training stats only -- never touches test data for fitting.
    '''
    filled  = df[NUMERIC_FEATURES].fillna(0).values
    zscores = np.zeros_like(filled)
    for sym in df['ExternalSymbol'].unique():
        mask      = (df['ExternalSymbol'] == sym).values
        sym_stats = stats.get(sym, stats['__global__'])  # use global if symbol is unseen
        for j, col in enumerate(NUMERIC_FEATURES):
            mu, sig          = sym_stats[col]
            zscores[mask, j] = np.abs((filled[mask, j] - mu) / sig)
    return zscores


def engineer_features(df, stats):
    '''Builds the full 37-feature matrix from the raw LOB columns.'''
    df = df.copy()
    df[NUMERIC_FEATURES] = df[NUMERIC_FEATURES].fillna(0)

    # z-scores for all 14 raw features
    zscores = compute_zscores(df, stats)
    df['max_zscore']  = zscores.max(axis=1)    # largest single-feature deviation
    df['mean_zscore'] = zscores.mean(axis=1)   # average deviation across all features
    for j, col in enumerate(NUMERIC_FEATURES):
        df[f'z_{col}'] = zscores[:, j]         # store each individual z-score too

    # simple magnitude features
    df['max_abs_feat']  = df[NUMERIC_FEATURES].abs().max(axis=1)
    df['abs_RetBid1']   = df['ReturnBid1'].abs()
    df['abs_RetAsk1']   = df['ReturnAsk1'].abs()
    df['return_spread'] = (df['ReturnBid1'] - df['ReturnAsk1']).abs()

    feat_cols = (
        NUMERIC_FEATURES
        + ['max_zscore', 'mean_zscore']
        + [f'z_{c}' for c in NUMERIC_FEATURES]
        + ['max_abs_feat', 'abs_RetBid1', 'abs_RetAsk1', 'return_spread']
        + ['burst_log', 'burst_10_49', 'burst_50plus']
    )
    return df, df[feat_cols].fillna(0).values


# --- run all feature engineering ---
print('Computing per-symbol statistics from normal data only...')
stats = compute_training_stats(normal)

print('Adding burst features...')
normal = add_burst_features(normal)
test   = add_burst_features(test)   # computed within test rows only, no cross-contamination

print('Building feature matrices...')
normal, X_normal = engineer_features(normal, stats)
test,   X_test   = engineer_features(test,   stats)

print()
print('Feature matrix shapes:')
print(f'  Normal : {X_normal.shape}  -- used to FIT models')
print(f'  Test   : {X_test.shape}   -- used only for SCORING')

## Section 4: Detector 1 (Z-Score Heuristic) and Detector 2 (Isolation Forest)

### Z-Score Heuristic

The simplest detector: for each test row, we check how extreme its features are relative to the normal training distribution (per stock symbol). We combine the z-score signals into one weighted anomaly score:

- `max_zscore` x 3 (log-transformed): the worst single-feature deviation
- `burst_10_49` x 8: our dominant fraud signal from EDA (175x higher returns)
- `burst_50plus` x 4: still strong, slightly less concentrated
- `abs_RetBid1` x 0.5: additional magnitude signal

No model is fit here, it's pure domain knowledge encoded as a formula, normalized to [0, 1].

### Isolation Forest (*not covered in class, explained from scratch*)

Isolation Forest (Liu et al., 2008) is an **ensemble anomaly detector** we hadn't seen in our course, so here is how it works.

#### Building the forest

We build 400 **isolation trees**. Each tree is constructed by:
1. Randomly picking one of the 37 features
2. Randomly picking a split threshold between the min and max of that feature in the current subset
3. Recursing on both halves until each point is isolated (alone) or a depth limit is hit

This is very different from the decision trees we saw in class (which choose splits to maximize information gain or minimize Gini impurity). Here, splits are **completely random** and the tree doesn't care about any target label.

#### Why random splitting detects anomalies

**Anomalies live in sparse regions of feature space.** With random splits, a sparse point gets isolated in just a few cuts, it has no neighbors to hide behind. Normal points live in dense regions and need many random cuts to be separated from the crowd.

The anomaly score for point $x$ given $n$ training samples is:

$$s(x, n) = 2^{-\dfrac{\mathbb{E}[h(x)]}{c(n)}}$$

where:
- $h(x)$ = depth at which $x$ gets isolated in one tree
- $\mathbb{E}[h(x)]$ = average path length across all 400 trees
- $c(n) = 2 H(n-1) - \dfrac{2(n-1)}{n}$ is a normalization constant ($H$ = harmonic number)

When $s \to 1$: very short average path → anomaly. When $s \to 0.5$: average path → normal point.

#### Why this fits our problem

We have no fraud labels, so we can't train a supervised model. Isolation Forest is **fully unsupervised**, it detects anomalies from geometric sparseness in feature space, not from labels. That's exactly the setting we're in.

We use `RobustScaler` before fitting, which scales by interquartile range rather than standard deviation. This avoids letting the few extreme values in the data distort the scaling for all the normal points.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import RobustScaler

# --- Detector 1: Z-Score Heuristic ---
# hand-crafted weighted sum of anomaly signals, normalized to [0, 1]
# weights reflect EDA findings: burst_10_49 = 8x because it was the dominant signal
z_sc   = (lambda df: (lambda s: (s-s.min())/(s.max()-s.min()+1e-9))(
    3.0*np.log1p(df['max_zscore'].values) + 1.0*np.log1p(df['mean_zscore'].values)
    + 8.0*df['burst_10_49'].values + 4.0*df['burst_50plus'].values
    + 1.0*df['burst_log'].values   + 0.5*np.log1p(df['abs_RetBid1'].values)))(test)

# --- Detector 2: Isolation Forest ---
# RobustScaler scales by IQR instead of std -- more robust to extreme outliers
# fit ONLY on normal data, then apply the same scaler to test without re-fitting
scaler = RobustScaler()
X_n_sc = scaler.fit_transform(X_normal)   # fit + transform on normal pool
X_t_sc = scaler.transform(X_test)         # transform only on test (no re-fitting)

# 400 trees, random subsamples of 512 rows per tree
# contamination=0.01 tells the model to expect roughly 1% anomalies (used internally)
iforest = IsolationForest(n_estimators=400, max_samples=512,
                          contamination=0.01, n_jobs=-1, random_state=42)
iforest.fit(X_n_sc)  # trained on normal data only

# score_samples returns NEGATIVE anomaly scores (more negative = more anomalous)
# we negate and normalize to [0, 1] so that higher score = more suspicious
if_sc = (lambda a: (a-a.min())/(a.max()-a.min()+1e-9))(-iforest.score_samples(X_t_sc))
print(f'z-score > 0.5: {(z_sc > 0.5).sum():,}  |  IF > 0.5: {(if_sc > 0.5).sum():,}')

## Section 5: Detector 3 - Dense Autoencoder (new in v2)

We covered both **autoencoders** and **MLP** in class, so this section focuses on *how we apply those concepts* to anomaly detection rather than re-deriving the full math.

### Quick recap from class

An autoencoder is a neural network trained to reproduce its own input:
- **Encoder**: compresses the input into a smaller bottleneck representation
- **Decoder**: reconstructs the original input from that compressed form

The training loss is mean squared reconstruction error:

$$\mathcal{L} = \frac{1}{n} \sum_{i=1}^{n} \|x_i - \hat{x}_i\|_2^2$$

The bottleneck forces the network to learn the most important structure in the data, it can't memorize everything, so it keeps the essential patterns.

### Our specific architecture

```
Input  (37 features)
  -> Dense(20, ReLU)    <- encoder layer 1
  -> Dense(8,  ReLU)    <- bottleneck  (compress 37 features down to 8)
  -> Dense(20, ReLU)    <- decoder layer 1
  -> Output (37)        <- reconstruction
```

8 bottleneck neurons for 37 input features. The network must learn a compressed representation of what *normal* LOB activity looks like.

### Why this catches fraud

We train **only on normal data**. The network learns the manifold of normal market patterns. When we feed it a fraud row, a pattern it has never seen, the decoder must reconstruct it using a compressed representation of *normal* behavior. Since fraud has different statistical structure, the reconstruction fails. **High MSE = row doesn't fit the normal manifold = anomaly.**

This is exactly the intuition from class: autoencoders learn a "normal manifold", and off-manifold points reconstruct poorly.

### Implementation note

We use sklearn's `MLPRegressor` with `hidden_layer_sizes=(20, 8, 20)` and call `fit(X, X)`, passing the same matrix as both input and target. This is the autoencoder trick from class, implemented via sklearn's MLP. No PyTorch or Keras needed, which keeps the environment simpler.

In [ ]:
print('Training Dense Autoencoder on normal data only...')
print('Architecture: 37 -> 20 -> 8 -> 20 -> 37')
print(f'Training samples: {len(X_n_sc):,}')

autoencoder = MLPRegressor(
    hidden_layer_sizes = (20, 8, 20),   # encoder: 37->20->8, decoder: 8->20->37
    activation         = 'relu',        # ReLU at hidden layers, same as class examples
    solver             = 'adam',        # Adam optimizer with adaptive learning rate
    learning_rate_init = 0.001,         # standard starting learning rate for Adam
    max_iter           = 50,            # 50 full passes through the training data
    batch_size         = 1024,          # mini-batch gradient descent
    random_state       = 42,
    verbose            = False,
)

# the autoencoder trick: input and target are the SAME matrix
# the network is forced to learn to compress and reconstruct its own input
autoencoder.fit(X_n_sc, X_n_sc)
print(f'Done.  Final reconstruction loss (MSE): {autoencoder.loss_:.6f}')

In [ ]:
# pass each test row through the trained autoencoder and measure how badly it reconstructs
X_recon     = autoencoder.predict(X_t_sc)                      # run test rows through the network
recon_error = np.mean((X_t_sc - X_recon) ** 2, axis=1)        # per-row MSE

# normalize to [0, 1] so this is on the same scale as z_sc and if_sc
ae_sc = (lambda a: (a-a.min())/(a.max()-a.min()+1e-9))(recon_error)

print('Dense Autoencoder score distribution:')
for t in [0.5, 0.7, 0.9]:
    print(f'  score > {t}: {(ae_sc > t).sum():,} rows')

# what does the AE catch that z-score alone misses?
# these are multivariate anomalies: no single feature is extreme, but the pattern as a whole is unusual
only_ae = (ae_sc > 0.7) & (z_sc <= 0.7)
print()
print(f'Rows only the Dense AE catches (AE > 0.7, z-score <= 0.7): {only_ae.sum():,}')
print('These are multivariate anomalies -- no single extreme feature, but the combination is unusual.')

In [ ]:
# look at the most anomalous row: actual features vs what the AE tried to reconstruct
worst_idx = np.argmax(recon_error)
actual    = X_t_sc[worst_idx]
recon     = X_recon[worst_idx]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
x = np.arange(len(NUMERIC_FEATURES))

# left: actual vs reconstructed for the single most suspicious row
# large gaps between red and blue bars = features the model reconstructed very badly
axes[0].bar(x - 0.2, actual[:len(NUMERIC_FEATURES)],  width=0.4, label='Actual',        color='#E24B4A', alpha=0.8)
axes[0].bar(x + 0.2, recon[:len(NUMERIC_FEATURES)],   width=0.4, label='Reconstructed', color='#4A90D9', alpha=0.8)
axes[0].set_xticks(x); axes[0].set_xticklabels(NUMERIC_FEATURES, rotation=40, ha='right', fontsize=8)
axes[0].set_title(f'Most anomalous row -- actual vs AE reconstruction  (MSE={recon_error[worst_idx]:.4f})')
axes[0].legend()

# right: scatter of z-score vs AE score for all test rows
# purple dots = rows the AE catches but z-score completely misses (top-left quadrant)
axes[1].scatter(z_sc[::10],    ae_sc[::10],    s=3, alpha=0.08, color='gray', label='all rows')
axes[1].scatter(z_sc[only_ae], ae_sc[only_ae], s=8, alpha=0.6,  color='#7F77DD', label=f'Only AE catches ({only_ae.sum():,})')
axes[1].axhline(0.7, color='red', linestyle='--', alpha=0.3)
axes[1].axvline(0.7, color='red', linestyle='--', alpha=0.3)
axes[1].set_xlabel('z-score'); axes[1].set_ylabel('Dense AE score')
axes[1].set_title('Purple = rows Dense AE catches that z-score misses')
axes[1].legend(markerscale=3)

plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}v2_ae_analysis.png', dpi=120, bbox_inches='tight')
plt.show()

## Section 6: Combining All Three Detectors into a Weighted Ensemble

We covered **ensemble methods** in class (bagging, boosting, random forests). This is a simpler variant: a fixed weighted average of three independently trained detectors.

- **Z-score heuristic** gets 50% of the weight because it directly encodes the burst signal, which is the dominant fraud indicator in this dataset
- **Isolation Forest** gets 25% of the weight; it catches geometric sparseness in 37-dimensional feature space
- **Dense Autoencoder** gets 25% of the weight; it catches multivariate patterns that deviate from the learned normal structure

The IF and AE each add something genuinely different: IF works on geometric isolation, AE works on reconstruction fidelity. No single detector catches everything, but together they cover more ground.

The final ensemble score is normalized to [0, 1] and submitted as the TARGET column.

In [ ]:
# weighted average of the three normalized scores
final = 0.50 * z_sc + 0.25 * if_sc + 0.25 * ae_sc
final = (lambda a: (a-a.min())/(a.max()-a.min()+1e-9))(final)  # re-normalize to [0, 1]

# plot all four distributions side-by-side (log y-axis to see the thin high-score tail)
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, score, label, color in zip(axes,
    [z_sc, if_sc, ae_sc, final],
    ['z-score', 'Isolation Forest', 'Dense AE', 'Ensemble v2'],
    ['#4A90D9', '#E88B2A', '#7F77DD', '#D85A30']
):
    ax.hist(score, bins=80, color=color, alpha=0.8, edgecolor='none')
    ax.set_title(label, fontsize=10); ax.set_yscale('log')
    ax.set_xlabel('Score [0,1]', fontsize=9)
    ax.axvline(0.9, color='black', linestyle='--', alpha=0.4)  # reference line at 0.9

plt.suptitle('v2: score distributions for all three detectors + ensemble', fontsize=11)
plt.tight_layout()
plt.savefig(f'{OUTPUT_DIR}v2_scores.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
def save_submission(scores, filename):
    '''Saves anomaly scores in Kaggle format: ID and TARGET columns.'''
    sub = pd.DataFrame({'ID': test['index'].values, 'TARGET': scores})
    path = f'{OUTPUT_DIR}{filename}'
    sub.to_csv(path, index=False)
    print(f'Saved: {path}')
    print(f'  Rows        : {len(sub):,}')
    print(f'  Score range : [{scores.min():.4f}, {scores.max():.4f}]')
    print(f'  Score > 0.9 : {(scores > 0.9).sum():,} flagged rows')
    return sub

sub = save_submission(final, 'submission_v2.csv')
print(sub.head())

## Section 7: Sanity Check - Are Train and Val Actually Separate?

Before trusting our combined normal pool, we verify that train and val do not overlap. If they shared rows, merging them would not give us more information and our whole setup would be questionable.

We check three things:
1. Do any row IDs appear in both files?
2. Are any rows byte-identical (same hash)?
3. Do the feature distributions look similar? (They should, if both came from the same market.)

In [ ]:
print('=' * 60)
print('TRAIN vs VAL IDENTITY CHECK')
print('=' * 60)

_train = pd.read_csv(TRAIN_PATH)
_val   = pd.read_csv(VAL_PATH)

print(f'  Train : {_train.shape[0]:,} rows x {_train.shape[1]} cols')
print(f'  Val   : {_val.shape[0]:,} rows x {_val.shape[1]} cols')

# check 1: same column names?
same_cols = set(_train.columns) == set(_val.columns)
print(f'  Same columns : {same_cols}')

# check 2: do any row IDs appear in both?
key = 'index' if 'index' in _train.columns else _train.columns[0]
print(f'  Key column   : {key!r}')
train_ids  = set(_train[key].astype(str))
val_ids    = set(_val[key].astype(str))
id_overlap = train_ids & val_ids
print(f'  Train unique IDs : {len(train_ids):,}')
print(f'  Val   unique IDs : {len(val_ids):,}')
print(f'  Shared IDs       : {len(id_overlap):,}')

# check 3: row-hash check -- catches cases where IDs differ but content is identical
n_sample    = min(50_000, len(_train), len(_val))
common_cols = [c for c in _train.columns if c in _val.columns]
t_hashes    = set(pd.util.hash_pandas_object(
    _train[common_cols].sample(n_sample, random_state=42)).values)
v_hashes    = set(pd.util.hash_pandas_object(
    _val[common_cols].sample(n_sample, random_state=42)).values)
hash_overlap = len(t_hashes & v_hashes)
print(f'  Row-hash overlap (sample n={n_sample:,}): {hash_overlap:,}')
print(f'  (greater than 0 means byte-identical rows exist in both sets)')

# check 4: how similar are the feature distributions?
print()
print('  Relative mean difference (train vs val) per feature:')
deltas = {}
for col in NUMERIC_FEATURES:
    if col in _train.columns and col in _val.columns:
        t_mu = _train[col].mean()
        v_mu = _val[col].mean()
        rel  = abs(t_mu - v_mu) / (abs(t_mu) + 1e-12)
        deltas[col] = rel
max_delta_col = max(deltas, key=deltas.get)
print(f'  Max relative mean delta : {deltas[max_delta_col]:.4f}  ({max_delta_col})')
print(f'  Avg relative mean delta : {sum(deltas.values())/len(deltas):.4f}')
print(f'  (Near 0 = distributions match, near 1 = very different)')

# verdict
print()
print('=' * 60)
if len(id_overlap) > 0:
    pct = len(id_overlap) / min(len(train_ids), len(val_ids)) * 100
    print(f'WARNING: {len(id_overlap):,} IDs shared ({pct:.1f}% of smaller set).')
    print('Train and val are NOT fully disjoint -- possible data leakage.')
elif hash_overlap > 0:
    print(f'WARNING: {hash_overlap:,} hash-identical rows found in sample.')
    print('Some rows may be duplicated across train and val.')
else:
    print('Train and val are distinct (no ID or row-hash overlap).')
    print('If the submission still scores below 1.0, the issue is model')
    print('sensitivity, not duplicate data between train and val.')
print('=' * 60)

del _train, _val  # free memory

## Section 8: AUROC - What It Means and How We Estimate It Without Test Labels

### What is AUROC?

**AUROC** = Area Under the Receiver Operating Characteristic Curve. It is the metric Kaggle uses to score us, and it is worth understanding what it actually measures.

The ROC curve sweeps a decision threshold from 0 to 1 and at each threshold plots:
- **True Positive Rate (TPR)**: what fraction of actual fraud rows did we flag?
- **False Positive Rate (FPR)**: what fraction of normal rows did we wrongly flag?

The **area** under this curve summarizes how well our model *ranks* fraud above normal across all possible thresholds:

- 1.0 means every fraud row scores higher than every normal row, which is perfect
- 0.5 means random guessing and the model learns nothing
- 0.88 is the class baseline we are trying to beat

AUROC is the right metric here because we care about *ranking*, not a specific cutoff. Our models output a continuous anomaly score and AUROC measures whether those scores consistently put fraud above normal.

### The problem: we cannot compute AUROC on the test set

Kaggle holds the fraud labels for the test set. We never see them. So we estimate performance three ways:

**1. False positive rate on val**: Score the validation set (all normal rows). Since these are genuinely normal, they should all score *low*. High scores on known-normal rows means we are producing false positives.

**2. Synthetic anomaly test**: We take 2,000 val rows (known normal) and corrupt them artificially by multiplying `ReturnBid1` and `ReturnAsk1` by a random factor of 10 to 50. This mimics extreme price moves that fraud produces. We measure AUROC on the combined set of normal val rows plus synthetic fraud. This gives a **conservative lower bound**, since if real fraud is even more extreme, our true AUROC is higher.

The scoring method used here is **Mahalanobis distance**, the multivariate generalization of the z-score. Instead of standardizing one feature at a time, it standardizes over the joint distribution accounting for correlations between features:

$$d_M(x) = \sqrt{(x - \mu)^T \, \Sigma^{-1} \, (x - \mu)}$$

**3. Score gap analysis**: If our test score distribution has a clean empty gap between the bulk of normal scores and the small cluster of high-scoring rows, that strongly suggests fraud is completely separated from normal, meaning AUROC close to 1.0.

In [ ]:
'''
AUROC Self-Evaluation
Three ways to estimate performance without the real test labels:
  1. False positive check: normal val rows should all score low
  2. Synthetic anomaly injection: fake fraud into val, measure AUROC
  3. Score gap analysis: check for a natural empty separator in test predictions
'''

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, RocCurveDisplay
from sklearn.preprocessing import RobustScaler
import warnings
warnings.filterwarnings('ignore')

TRAIN_PATH = 'train_data.csv'
VAL_PATH   = 'val_data.csv'
TEST_PATH  = 'test_data.csv'
SUB_PATH   = 'submission_v4_continuous.csv'   # change to your actual submission file

NUMERIC_FEATURES = [
    'ReturnBid1', 'ReturnAsk1',
    'DerivativeReturnBid1', 'DerivativeReturnAsk1',
    'BidSize1', 'AskSize1', 'TradeBidSize', 'TradeAskSize',
    'CancelledBidSize', 'CancelledAskSize',
    'TradeBidIndicator', 'TradeAskIndicator',
    'CancelledBidIndicator', 'CancelledAskIndicator',
]

print('Loading data...')
train  = pd.read_csv(TRAIN_PATH)
val    = pd.read_csv(VAL_PATH)
test   = pd.read_csv(TEST_PATH)
sub    = pd.read_csv(SUB_PATH)
normal = pd.concat([train, val], ignore_index=True)
print(f'  Normal: {len(normal):,}  |  Test: {len(test):,}')
print(f'  Submission rows: {len(sub):,}')


def score_with_mahalanobis(df_score, df_normal):
    '''
    Per-symbol Mahalanobis distance:
      d_M(x) = sqrt( (x - mu)^T * Sigma^{-1} * (x - mu) )
    Fit on df_normal, score df_score. No test data used for fitting.
    '''
    scores = np.zeros(len(df_score))
    for sym in df_score['ExternalSymbol'].unique():
        nm = (df_normal['ExternalSymbol'] == sym).values
        sm = (df_score['ExternalSymbol']  == sym).values
        if nm.sum() < 20: continue  # skip symbols with too few normal examples
        Xn = df_normal[nm][NUMERIC_FEATURES].fillna(0).values.astype('float64')
        Xs = df_score[sm][NUMERIC_FEATURES].fillna(0).values.astype('float64')
        sc  = RobustScaler().fit(Xn)
        Xns = sc.transform(Xn)
        Xss = sc.transform(Xs)
        mu  = Xns.mean(0)
        cov = np.cov(Xns.T) + np.eye(14) * 1e-6  # small diagonal for numerical stability
        ci  = np.linalg.pinv(cov)                 # pseudo-inverse (more stable than linalg.inv)
        d   = Xss - mu
        scores[sm] = np.einsum('ij,jk,ik->i', d, ci, d)  # vectorized Mahalanobis distance
    return scores


# --- check 1: val rows (all normal) should score low ---
print()
print('[1/3] Sanity check -- scoring NORMAL val rows with Mahalanobis distance...')
print('      We fit on train and score val. Both are normal, so all scores should be low.')
val_scores      = score_with_mahalanobis(val, train)
val_scores_norm = (val_scores - val_scores.min()) / (val_scores.max() - val_scores.min() + 1e-9)

print('  Val anomaly score percentiles (all rows are genuinely normal):')
for p in [50, 90, 95, 99, 99.9, 100]:
    print(f'    p{p:5.1f}: {np.percentile(val_scores_norm, p):.4f}')

false_positive_rate = (val_scores_norm > 0.5).mean()
print(f'  False positive rate at threshold 0.5: {false_positive_rate:.2%}')
if false_positive_rate < 0.02:
    print('  Good -- model is not over-flagging normal rows')
else:
    print('  Warning -- model is flagging too many normal rows as fraud')


# --- check 2: inject synthetic fraud, measure AUROC ---
print()
print('[2/3] Synthetic anomaly injection test...')
print('      Taking 2000 normal val rows and multiplying returns by 10-50x to simulate fraud.')

np.random.seed(42)
n_synthetic = 2000
idx         = np.random.choice(len(val), n_synthetic, replace=False)
synthetic   = val.iloc[idx].copy()

# simulate extreme price moves, like what pump-and-dump produces
scale = np.random.uniform(10, 50, n_synthetic)
synthetic[NUMERIC_FEATURES] = synthetic[NUMERIC_FEATURES].fillna(0)
synthetic['ReturnBid1'] = synthetic['ReturnBid1'] * scale
synthetic['ReturnAsk1'] = synthetic['ReturnAsk1'] * scale * np.random.uniform(0.8, 1.2, n_synthetic)

eval_df     = pd.concat([val.reset_index(drop=True), synthetic.reset_index(drop=True)], ignore_index=True)
eval_labels = np.array([0] * len(val) + [1] * n_synthetic)

eval_scores      = score_with_mahalanobis(eval_df, train)
eval_scores_norm = (eval_scores - eval_scores.min()) / (eval_scores.max() - eval_scores.min() + 1e-9)

auroc_synthetic = roc_auc_score(eval_labels, eval_scores_norm)
print(f'  AUROC on normal val + synthetic anomalies: {auroc_synthetic:.4f}')
print(f'  (Conservative lower bound -- real fraud may be even more separable)')


# --- check 3: gap analysis ---
print()
print('[3/3] Score gap analysis on Kaggle submission...')
pred_scores = sub['TARGET'].values

print('  Submission score distribution:')
for p in [50, 90, 95, 99, 99.5, 99.9, 100]:
    print(f'    p{p:5.1f}: {np.percentile(pred_scores, p):.5f}')

# scan for the first empty bin above the dense normal mass
hist, edges = np.histogram(pred_scores, bins=2000)
gap = None
for i in range(len(hist)):
    if edges[i] > 0.05 and hist[i] == 0:
        gap = (edges[i], edges[i+1])
        break

if gap:
    n_above = (pred_scores > gap[0]).sum()
    print(f'  Natural gap found: [{gap[0]:.4f}, {gap[1]:.4f}]')
    print(f'  Rows above gap (predicted fraud): {n_above:,}')
    print(f'  A clean gap here strongly suggests AUROC close to 1.0')
else:
    print('  No clean gap found -- scores form a continuous distribution')


# --- plot all three checks ---
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

axes[0].hist(val_scores_norm, bins=80, color='#4A90D9', alpha=0.8)
axes[0].set_title('Normal (val) scores -- all should be low', fontsize=10)
axes[0].set_xlabel('Anomaly score [0, 1]')
axes[0].set_yscale('log')
axes[0].axvline(0.5, color='red', linestyle='--', linewidth=1, label='threshold=0.5')
axes[0].legend(fontsize=9)

RocCurveDisplay.from_predictions(eval_labels, eval_scores_norm, ax=axes[1],
                                  name=f'Synthetic test (AUROC={auroc_synthetic:.3f})',
                                  color='#0F6E56')
axes[1].plot([0,1],[0,1],'--',color='gray',linewidth=0.8)
axes[1].set_title('ROC curve -- synthetic anomaly test', fontsize=10)

axes[2].hist(pred_scores, bins=150, color='#D85A30', alpha=0.8)
if gap:
    axes[2].axvspan(gap[0], gap[1], alpha=0.3, color='green', label=f'Gap @ {gap[0]:.3f}')
    axes[2].legend(fontsize=9)
axes[2].set_title('Kaggle submission scores (test set)', fontsize=10)
axes[2].set_xlabel('Anomaly score [0, 1]')
axes[2].set_yscale('log')

plt.suptitle('AUROC self-evaluation -- three checks without test labels', fontsize=11)
plt.tight_layout()
plt.savefig('auroc_self_evaluation.png', dpi=120, bbox_inches='tight')
plt.show()

print()
print('=' * 55)
print('Summary')
print('=' * 55)
print(f'  False positive rate on normal data : {false_positive_rate:.2%}')
print(f'  AUROC on synthetic anomaly test    : {auroc_synthetic:.4f}')
print(f'  Natural gap in test predictions    : {"YES" if gap else "NO"}')
print()
print('How to read these:')
print('  FP rate < 2%           ->  model is not over-flagging normal rows')
print('  Synthetic AUROC > 0.95 ->  model separates anomalies well in principle')
print('  Gap exists             ->  AUROC on real test data is likely close to 1.0')
print()
print('Submit to Kaggle to get the actual AUROC on the real test labels.')